## Notebook04

### Setup

Run all of the following before starting the notebook.

In [5]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [6]:
import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

ub = "https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/"

In [7]:
film = (
    pl.read_csv(ub + "data/criterion.csv")
    .select(
        c.title, c.year, c.director, c.country, c.runtime_raw,
        c.imdb_votes, c.rating_imdb, c.rating_rt, c.budget_raw, c.box_office_raw
    )
    .drop_nulls(subset=["runtime_raw", "imdb_votes", "rating_imdb"])
)

### Questions

We will keep working with the Criterion Collection films from last time. The
dataset has two new columns: `budget_raw`, what the film cost to make, and
`box_office_raw`, what it earned. As before, print your results rather than
saving them.

1. The column `runtime_raw` has an awkward name. Rename it to `runtime`.

In [4]:
(
    film
    .rename({"runtime_raw" : "runtime"})
)

title,year,director,country,runtime,imdb_votes,rating_imdb,rating_rt,budget_raw,box_office_raw
str,i64,str,str,i64,i64,f64,i64,i64,i64
"""The Kid""",1921,"""Charles Chaplin""","""United States""",68,142797,8.2,100,250000,null
"""The Phantom Carriage""",1921,"""Victor Sjöström""","""Sweden""",106,15311,8.0,100,null,null
"""Häxan""",1922,"""Benjamin Christensen""","""Sweden|Denmark""",107,18391,7.6,93,2000000,null
"""Safety Last!""",1923,"""Fred C. Newmeyer|Sam Taylor""","""United States""",73,23503,8.1,97,121000,null
"""A Woman of Paris: A Drama of F…",1923,"""Charles Chaplin""","""United States""",84,6548,6.9,94,351000,634000
…,…,…,…,…,…,…,…,…,…
"""Caught by the Tides""",2024,"""Jia Zhang-ke""","""China|France|Japan""",111,2097,6.7,99,null,null
"""Anora""",2024,"""Sean Baker""","""United States""",139,235046,7.4,93,null,null
"""Vermiglio""",2024,"""Maura Delpero""","""Italy|France|Belgium""",119,4838,6.9,93,null,null


2. Print one film from each country in the collection.

In [5]:
(
    film
    .unique(subset="country")
)

title,year,director,country,runtime_raw,imdb_votes,rating_imdb,rating_rt,budget_raw,box_office_raw
str,i64,str,str,i64,i64,f64,i64,i64,i64
"""Once Upon a Time in China III""",1992,"""Hark Tsui""","""Hong Kong|China""",109,8025,6.7,63,null,null
"""The Passion of Joan of Arc""",1928,"""Carl Theodor Dreyer""","""France""",82,66599,8.1,98,null,null
"""Pat Garrett & Billy the Kid""",1973,"""Sam Peckinpah""","""United States|Mexico""",122,22809,7.2,60,null,null
"""Personal Shopper""",2016,"""Olivier Assayas""","""France|Germany|Czech Republic|…",105,45007,6.1,81,null,null
"""Walkabout""",1971,"""Nicolas Roeg""","""United Kingdom|Australia|Unite…",100,27979,7.6,86,1000000,null
…,…,…,…,…,…,…,…,…,…
"""The Shrouds""",2024,"""David Cronenberg""","""Canada|France""",120,4895,5.8,75,null,null
"""Limit""",1931,"""Mario Peixoto""","""Brazil""",120,2970,7.0,null,null,null
"""Oedipus Rex""",1967,"""Pier Paolo Pasolini""","""Italy|Morocco""",104,7061,7.2,89,null,null


3. The row you get back for each country is an arbitrary one. Modify the
previous answer so that the film you see from each country is its *earliest*
film. Print only the country, title, and year.

In [11]:
(
    film
    .sort("year")
    .unique(subset=["country"], keep="first")
    .select(c.country, c.title, c.year)
)

country,title,year
str,str,i64
"""Germany|United States""","""The Threepenny Opera""",1931
"""West Germany|United Kingdom""","""Autumn Sonata""",1978
"""Angola|France""","""Sambizanga""",1972
"""Austria|Germany""","""71 Fragments of a Chronology o…",1994
"""United Kingdom|Germany|United …","""Ghost World""",2001
…,…,…
"""Romania|France|Belgium""","""Beyond the Hills""",2012
"""United States|United Kingdom|M…","""El Norte""",1983
"""Italy|France|West Germany""","""I Knew Her Well""",1965


4. Create two new columns at the same time: `hours`, the runtime measured
in
hours rather than minutes, and `votes_1k`, the number of IMDb votes measured in
thousands.

In [11]:
(
    film
    .with_columns(hours = c.runtime_raw / 60, votes_1k = c.imdb_votes / 1000)
)

title,year,director,country,runtime_raw,imdb_votes,rating_imdb,rating_rt,budget_raw,box_office_raw,hours,votes_1k
str,i64,str,str,i64,i64,f64,i64,i64,i64,f64,f64
"""The Kid""",1921,"""Charles Chaplin""","""United States""",68,142797,8.2,100,250000,null,1.133333,142.797
"""The Phantom Carriage""",1921,"""Victor Sjöström""","""Sweden""",106,15311,8.0,100,null,null,1.766667,15.311
"""Häxan""",1922,"""Benjamin Christensen""","""Sweden|Denmark""",107,18391,7.6,93,2000000,null,1.783333,18.391
"""Safety Last!""",1923,"""Fred C. Newmeyer|Sam Taylor""","""United States""",73,23503,8.1,97,121000,null,1.216667,23.503
"""A Woman of Paris: A Drama of F…",1923,"""Charles Chaplin""","""United States""",84,6548,6.9,94,351000,634000,1.4,6.548
…,…,…,…,…,…,…,…,…,…,…,…
"""Caught by the Tides""",2024,"""Jia Zhang-ke""","""China|France|Japan""",111,2097,6.7,99,null,null,1.85,2.097
"""Anora""",2024,"""Sean Baker""","""United States""",139,235046,7.4,93,null,null,2.316667,235.046
"""Vermiglio""",2024,"""Maura Delpero""","""Italy|France|Belgium""",119,4838,6.9,93,null,null,1.983333,4.838


5. Create a column `decade` holding the decade the film was released in, so that
1957 becomes 1950 and 1962 becomes 1960. Hint: Python's `//` operator divides
and throws away the remainder, so `1957 // 10` is `195`.

In [12]:
(
    film
    .with_columns(decade = (c.year // 10) * 10)
)

title,year,director,country,runtime_raw,imdb_votes,rating_imdb,rating_rt,budget_raw,box_office_raw,decade
str,i64,str,str,i64,i64,f64,i64,i64,i64,i64
"""The Kid""",1921,"""Charles Chaplin""","""United States""",68,142797,8.2,100,250000,null,1920
"""The Phantom Carriage""",1921,"""Victor Sjöström""","""Sweden""",106,15311,8.0,100,null,null,1920
"""Häxan""",1922,"""Benjamin Christensen""","""Sweden|Denmark""",107,18391,7.6,93,2000000,null,1920
"""Safety Last!""",1923,"""Fred C. Newmeyer|Sam Taylor""","""United States""",73,23503,8.1,97,121000,null,1920
"""A Woman of Paris: A Drama of F…",1923,"""Charles Chaplin""","""United States""",84,6548,6.9,94,351000,634000,1920
…,…,…,…,…,…,…,…,…,…,…
"""Caught by the Tides""",2024,"""Jia Zhang-ke""","""China|France|Japan""",111,2097,6.7,99,null,null,2020
"""Anora""",2024,"""Sean Baker""","""United States""",139,235046,7.4,93,null,null,2020
"""Vermiglio""",2024,"""Maura Delpero""","""Italy|France|Belgium""",119,4838,6.9,93,null,null,2020


6. Rotten Tomatoes scores run from 0 to 100 and IMDb ratings from 0 to 10, so
dividing the first by ten puts them on the same scale. Build a column `gap` that
subtracts the IMDb rating from the rescaled Rotten Tomatoes score. Then print
the five films with the largest gap, showing only the title, year, and gap. A
large positive gap means the critics liked the film much more than the audience
did.

In [27]:
(
  film
  .with_columns(gap = (c.rating_rt/10) - c.rating_imdb)
  .drop_nulls("gap")
  .sort("gap", descending=True)
  .head(5)
  .select(c.title, c.year, c.gap)
)

title,year,gap
str,i64,f64
"""Jubilee""",1978,4.1
"""Elephant Boy""",1937,3.6
"""Totally F***ed Up""",1993,3.6
"""Thirst""",1949,3.5
"""The Shooting""",1966,3.5


Sorting in the other direction gives the films the audience preferred. Try it.

7. Create a column `profit` equal to the box office earnings minus the budget,
and print the five most profitable films (title, year, and profit only).

In [34]:
(
    film
    .with_columns(profit = c.box_office_raw - c.budget_raw)
    .drop_nulls("profit")
    .sort("profit", descending=True)
    .head(5)
    .select(c.title, c.year, c.profit)
)

title,year,profit
str,i64,i64
"""Yojimbo""",1961,260130000
"""High and Low""",1963,230200000
"""The Others""",2001,192900000
"""The Shape of Water""",2017,174849971
"""Tootsie""",1982,156200000


Take these numbers with a grain of salt. The budget and box office figures come
from IMDb, where contributors report them in the local currency of the
production, so the yen figures for the Japanese films at the top of this list
are not comparable to the dollar figures elsewhere in the column. A column of
numbers can be perfectly clean and still be measuring several different things.

8. Create an indicator column named `is_long` that is equal to 1 when a film
runs longer than two hours and 0 otherwise.

In [36]:
(
    film
    .with_columns(is_long = pl.when(c.runtime_raw > 120).then(1).otherwise(0))
)

title,year,director,country,runtime_raw,imdb_votes,rating_imdb,rating_rt,budget_raw,box_office_raw,is_long
str,i64,str,str,i64,i64,f64,i64,i64,i64,i32
"""The Kid""",1921,"""Charles Chaplin""","""United States""",68,142797,8.2,100,250000,null,0
"""The Phantom Carriage""",1921,"""Victor Sjöström""","""Sweden""",106,15311,8.0,100,null,null,0
"""Häxan""",1922,"""Benjamin Christensen""","""Sweden|Denmark""",107,18391,7.6,93,2000000,null,0
"""Safety Last!""",1923,"""Fred C. Newmeyer|Sam Taylor""","""United States""",73,23503,8.1,97,121000,null,0
"""A Woman of Paris: A Drama of F…",1923,"""Charles Chaplin""","""United States""",84,6548,6.9,94,351000,634000,0
…,…,…,…,…,…,…,…,…,…,…
"""Caught by the Tides""",2024,"""Jia Zhang-ke""","""China|France|Japan""",111,2097,6.7,99,null,null,0
"""Anora""",2024,"""Sean Baker""","""United States""",139,235046,7.4,93,null,null,1
"""Vermiglio""",2024,"""Maura Delpero""","""Italy|France|Belgium""",119,4838,6.9,93,null,null,0


9. Now build a column named `era` that labels each film `"classic"` if it was
released before 1950, `"modern"` if it was released before 1990, and
`"contemporary"` otherwise. Print only the title, year, and era. This needs a
chained `.when().then()` pair and the `pl.lit` from the climate zone example in
the chapter.

In [45]:
(
    film
    .with_columns(
        era =
          pl.when(c.year < 1950).then(pl.lit("classic"))
          .when((c.year <= 1990) & (c.year > 1950)).then(pl.lit("modern"))
          .otherwise(pl.lit("contemporary")))
    .select(c.title, c.year, c.era)
)

title,year,era
str,i64,str
"""The Kid""",1921,"""classic"""
"""The Phantom Carriage""",1921,"""classic"""
"""Häxan""",1922,"""classic"""
"""Safety Last!""",1923,"""classic"""
"""A Woman of Paris: A Drama of F…",1923,"""classic"""
…,…,…
"""Caught by the Tides""",2024,"""contemporary"""
"""Anora""",2024,"""contemporary"""
"""Vermiglio""",2024,"""contemporary"""


Because the conditions are checked in order, the second one only ever sees the
films the first did not already claim, which is why `c.year < 1990` does not need
to say anything about 1950.

10. Finally, print *every* film that runs longer than 200 minutes, showing the
title, director, and runtime. There are eleven of them, and Polars hides the
middle of any table longer than a handful of rows, so end the chain with the
`.print()` method from `funs.py` and ask it for eleven rows.

In [8]:
(
  film
  .filter(c.runtime_raw > 200)
  .select(c.title, c.director, c.runtime_raw)
  .print(12)
)

title,director,runtime_raw
str,str,i64
"""Les Misérables""","""Raymond Bernard""",281
"""Seven Samurai""","""Akira Kurosawa""",207
"""War and Peace""","""Sergey Bondarchuk""",432
"""The New Land""","""Jan Troell""",204
"""The Mother and the Whore""","""Jean Eustache""",217
"""Jeanne Dielman, 23, quai du Co…","""Chantal Akerman""",201
"""Shoah""","""Claude Lanzmann""",540
"""A Brighter Summer Day""","""Edward Yang""",237
"""Malcolm X""","""Spike Lee""",201


`.print()` is a method we wrote ourselves in `funs.py` rather than one that comes
with Polars. It is the last step of the chain because it displays the table and
hands nothing back, so there is nothing left to add a method to.